# Mapeamento Eleitoral e Visualização Geospacial com Folium

**Dupla:** Nathan Mateus de Lima e Vítor Oliveira Amorim  
**Professora:** Carolina Ribeiro Xavier  
**Disciplina:** Tecnologias para Data Science  

---

In [46]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import pathlib
import math

from folium.plugins import Fullscreen

**Caminho até o csv com os dados**

In [47]:
data_path = pathlib.Path('./data')
sjdr_votacao_csv_path = data_path / 'votacao_sjdr.csv' 

In [48]:
df = pd.read_csv(sjdr_votacao_csv_path, sep=',')
df.head()

,CD_MUNICIPIO,NR_ZONA,NR_SECAO,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,...,DS_SITU_LOCALIDADE,CD_SITU_SECAO_ACESSIBILIDADE,DS_SITU_SECAO_ACESSIBILIDADE,QT_ELEITOR_SECAO,QT_ELEITOR_ELEICAO_FEDERAL,QT_ELEITOR_ELEICAO_ESTADUAL,QT_ELEITOR_ELEICAO_MUNICIPAL,NR_LOCAL_VOTACAO_ORIGINAL,NM_LOCAL_VOTACAO_ORIGINAL,DS_ENDERECO_LOCVT_ORIGINAL
0,52493,328,52,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,...,Ativo,0.0,Sem acessibilidade,238.0,415.0,415.0,0.0,1023.0,COLEGIO NOSSA SENHORA DAS DORES,"RUA COMENDADOR BASTOS, 84"
1,52493,328,224,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,...,Ativo,0.0,Sem acessibilidade,323.0,325.0,325.0,0.0,1716.0,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,"PRAÇA BOM JESUS DE MATOZINHOS, S/N"
2,52493,328,324,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,...,Ativo,0.0,Sem acessibilidade,359.0,356.0,356.0,0.0,1864.0,ESCOLA ESTADUAL GOVERNADOR MILTON CAMPOS,"AV. SETE DE SETEMBRO, S/N"
3,52493,328,169,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,...,Ativo,0.0,Sem acessibilidade,282.0,281.0,281.0,0.0,1449.0,ESCOLA ESTADUAL PADRE LOPES,FRANCISCO DAS CHAGAS RIBEIRO - 520
4,52493,328,50,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,...,Ativo,0.0,Sem acessibilidade,237.0,411.0,411.0,0.0,1023.0,COLEGIO NOSSA SENHORA DAS DORES,"RUA COMENDADOR BASTOS, 84"


In [49]:
df['NR_LATITUDE'] = pd.to_numeric(
    df['NR_LATITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
).round(4)
df['NR_LONGITUDE'] = pd.to_numeric(
    df['NR_LONGITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
).round(4)

## Análises Preliminares dos Dados

In [50]:
df_turno1 = df[df["NR_TURNO"] == 1]
df_turno2 = df[df["NR_TURNO"] == 2]

count_turno1 = len(df_turno1)
count_turno2 = len(df_turno2)

print("Qtd de registros turno 1:", count_turno1)
print("Qtd de registros turno 2:", count_turno2)

Qtd de registros turno 1: 1851
Qtd de registros turno 2: 855


**Conta a quantidade de zonas eleitorais**

SJDR tem somente uma zona eleitoral (zona 328)

In [51]:
df_zona_1T = df_turno1.groupby('NR_ZONA')
df_zona_2T = df_turno2.groupby('NR_ZONA')

print(f'Zonas 1T: {len(df_zona_1T)}') 
print(f'Zonas 2T: {len(df_zona_2T)}')

Zonas 1T: 1
Zonas 2T: 1


**Conta a quantidade de secoes eleitorais**

A zona 328 tem 241 secoes eleitorais, contudo ela abrange cidades da regiao de SJDR, nao somente SJDR. Como filtramos apenas por SJDJ, e normal que tenham menos secoes que o total de secoes da zona.

In [52]:
secoes_1T = df_zona_1T['NR_SECAO'].nunique() 
secoes_2T = df_zona_2T['NR_SECAO'].nunique() 

print(f"Qtd secoes 1T: {secoes_1T.iloc[0]}")
print(f"Qtd secoes 2T: {secoes_2T.iloc[0]}")

Qtd secoes 1T: 218
Qtd secoes 2T: 218


## Limpeza, Filtragem e Unificação dos Dados

### Agrupa o Total de Votos por Candidato em Ambos os Turnos 

Agrupamentos feitos por **Local de Votação**, **Seções** e **Bairros**

**Agrupa por Locais de Votação**

In [53]:
# Seleciona o nome do local com mais eleitores para cada par [lat, lon]
nm_local_principal_1T = (
    df_turno1.groupby(['NR_LATITUDE', 'NR_LONGITUDE', 'NM_LOCAL_VOTACAO'], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by='QT_VOTOS', ascending=False)
        .drop_duplicates(subset=['NR_LATITUDE', 'NR_LONGITUDE'])
        [['NR_LATITUDE', 'NR_LONGITUDE', 'NM_LOCAL_VOTACAO']]
)

# Agrupa os votos por [lat, lon] e candidato, e traz o nome do local de votacao obtido anteriormente
df_locais_1T = (
    df_turno1.groupby(['NR_LATITUDE', 'NR_LONGITUDE', 'NM_VOTAVEL', "NR_VOTAVEL"], as_index=False)['QT_VOTOS']
        .sum()
        .merge(nm_local_principal_1T, on=['NR_LATITUDE', 'NR_LONGITUDE'], how='left')
        .sort_values(by=['NM_LOCAL_VOTACAO', 'QT_VOTOS'], ascending=[True, False],)
)

# Calcula o percentual de votos de cada candidato por local 
total_votos_local1T = df_locais_1T.groupby(['NR_LATITUDE', 'NR_LONGITUDE'])['QT_VOTOS'].transform('sum')
df_locais_1T['PCT_VOTOS_LOCAL'] = (df_locais_1T['QT_VOTOS'] / total_votos_local1T) * 100.00
df_locais_1T['PCT_VOTOS_LOCAL'] = df_locais_1T['PCT_VOTOS_LOCAL'].round(2)

df_locais_1T.head(10)

,NR_LATITUDE,NR_LONGITUDE,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,NM_LOCAL_VOTACAO,PCT_VOTOS_LOCAL
466,-21.1207,-44.2465,FLAVIO BOLSONARO,22,347,ANTIGO PREDIO DO UNIPTAN-FUNDOS,45.66
467,-21.1207,-44.2465,LULA,13,305,ANTIGO PREDIO DO UNIPTAN-FUNDOS,40.13
465,-21.1207,-44.2465,ESCRITOR AUGUSTO CURY,70,29,ANTIGO PREDIO DO UNIPTAN-FUNDOS,3.82
468,-21.1207,-44.2465,Nulo,96,23,ANTIGO PREDIO DO UNIPTAN-FUNDOS,3.03
469,-21.1207,-44.2465,RENAN SANTOS,14,16,ANTIGO PREDIO DO UNIPTAN-FUNDOS,2.11
463,-21.1207,-44.2465,Branco,95,15,ANTIGO PREDIO DO UNIPTAN-FUNDOS,1.97
470,-21.1207,-44.2465,RONALDO CAIADO,55,13,ANTIGO PREDIO DO UNIPTAN-FUNDOS,1.71
472,-21.1207,-44.2465,ZEMA,30,8,ANTIGO PREDIO DO UNIPTAN-FUNDOS,1.05
464,-21.1207,-44.2465,CLARIANA BARAO,27,2,ANTIGO PREDIO DO UNIPTAN-FUNDOS,0.26
471,-21.1207,-44.2465,SAMARA,80,2,ANTIGO PREDIO DO UNIPTAN-FUNDOS,0.26


In [65]:
# Seleciona o nome do local com mais eleitores para cada par [lat, lon]
nm_local_principal_2T = (
    df_turno2.groupby(['NR_LATITUDE', 'NR_LONGITUDE', 'NM_LOCAL_VOTACAO'], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by='QT_VOTOS', ascending=False)
        .drop_duplicates(subset=['NR_LATITUDE', 'NR_LONGITUDE'])
        [['NR_LATITUDE', 'NR_LONGITUDE', 'NM_LOCAL_VOTACAO']]
)

# Agrupa os votos por [lat, lon] e candidato, e traz o nome do local de votacao obtido anteriormente
df_locais_2T = (
    df_turno2.groupby(['NR_LATITUDE', 'NR_LONGITUDE', 'NM_VOTAVEL', "NR_VOTAVEL"], as_index=False)['QT_VOTOS']
        .sum()
        .merge(nm_local_principal_2T, on=['NR_LATITUDE', 'NR_LONGITUDE'], how='left')
        .sort_values(by=['NM_LOCAL_VOTACAO', 'QT_VOTOS'], ascending=[True, False],)
)

# Calcula o percentual de votos de cada candidato por local 
total_votos_local2T = df_locais_2T.groupby(['NR_LATITUDE', 'NR_LONGITUDE'])['QT_VOTOS'].transform('sum')
df_locais_2T['PCT_VOTOS_LOCAL'] = (df_locais_2T['QT_VOTOS'] / total_votos_local2T) * 100.00
df_locais_2T['PCT_VOTOS_LOCAL'] = df_locais_2T['PCT_VOTOS_LOCAL'].round(2)

df_locais_2T.head(10)

,NR_LATITUDE,NR_LONGITUDE,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,NM_LOCAL_VOTACAO,PCT_VOTOS_LOCAL
198,-21.1207,-44.2465,FLAVIO BOLSONARO,22,402,ANTIGO PREDIO DO UNIPTAN-FUNDOS,50.38
199,-21.1207,-44.2465,LULA,13,367,ANTIGO PREDIO DO UNIPTAN-FUNDOS,45.99
200,-21.1207,-44.2465,Nulo,96,20,ANTIGO PREDIO DO UNIPTAN-FUNDOS,2.51
197,-21.1207,-44.2465,Branco,95,9,ANTIGO PREDIO DO UNIPTAN-FUNDOS,1.13
222,-21.1050,-44.3427,LULA,13,122,ASSOCIAÇÃO DE MORADORES E PRODUTORES RURAIS DE...,50.62
221,-21.1050,-44.3427,FLAVIO BOLSONARO,22,111,ASSOCIAÇÃO DE MORADORES E PRODUTORES RURAIS DE...,46.06
220,-21.1050,-44.3427,Branco,95,4,ASSOCIAÇÃO DE MORADORES E PRODUTORES RURAIS DE...,1.66
223,-21.1050,-44.3427,Nulo,96,4,ASSOCIAÇÃO DE MORADORES E PRODUTORES RURAIS DE...,1.66
123,-21.1372,-44.2657,LULA,13,163,BANDA TEODORO DE FARIA,49.24
122,-21.1372,-44.2657,FLAVIO BOLSONARO,22,160,BANDA TEODORO DE FARIA,48.34


**Agrupa por seção**

In [55]:
df_secoes_1T = (
    df_turno1.groupby(['NR_SECAO', 'NM_VOTAVEL', "NR_VOTAVEL"], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NR_SECAO', 'QT_VOTOS'], ascending=[True, False],)
)

# Calcula o percentual de votos de cada candidato por secao 
total_votos_secao1T = df_secoes_1T.groupby('NR_SECAO')['QT_VOTOS'].transform('sum')
df_secoes_1T['PCT_VOTOS_SECAO'] = (df_secoes_1T['QT_VOTOS'] / total_votos_secao1T) * 100.00
df_secoes_1T['PCT_VOTOS_SECAO'] = df_secoes_1T['PCT_VOTOS_SECAO'].round(2)

df_secoes_1T.head(10)

,NR_SECAO,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,PCT_VOTOS_SECAO
2,40,FLAVIO BOLSONARO,22,122,53.04
3,40,LULA,13,85,36.96
4,40,Nulo,96,9,3.91
6,40,RONALDO CAIADO,55,4,1.74
1,40,ESCRITOR AUGUSTO CURY,70,3,1.30
5,40,RENAN SANTOS,14,3,1.30
0,40,Branco,95,2,0.87
7,40,ZEMA,30,2,0.87
10,41,FLAVIO BOLSONARO,22,119,49.58
12,41,LULA,13,91,37.92


In [56]:
df_secoes_2T = (
    df_turno2.groupby(['NR_SECAO', 'NM_VOTAVEL', "NR_VOTAVEL"], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NR_SECAO', 'QT_VOTOS'], ascending=[True, False],)
)

# Calcula o percentual de votos de cada candidato por secao 
total_votos_secao2T = df_secoes_2T.groupby('NR_SECAO')['QT_VOTOS'].transform('sum')
df_secoes_2T['PCT_VOTOS_SECAO'] = (df_secoes_2T['QT_VOTOS'] / total_votos_secao2T) * 100.00
df_secoes_2T['PCT_VOTOS_SECAO'] = df_secoes_2T['PCT_VOTOS_SECAO'].round(2)

df_secoes_2T.head(10)

,NR_SECAO,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,PCT_VOTOS_SECAO
1,40,FLAVIO BOLSONARO,22,130,56.03
2,40,LULA,13,95,40.95
3,40,Nulo,96,6,2.59
0,40,Branco,95,1,0.43
5,41,FLAVIO BOLSONARO,22,136,53.97
6,41,LULA,13,109,43.25
7,41,Nulo,96,4,1.59
4,41,Branco,95,3,1.19
9,43,FLAVIO BOLSONARO,22,109,56.48
10,43,LULA,13,80,41.45


**Agrupa por Bairro**

In [57]:
df_bairros_1T = (
    df_turno1.groupby(['NM_BAIRRO', 'NM_VOTAVEL', "NR_VOTAVEL"], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NM_BAIRRO', 'QT_VOTOS'], ascending=[True, False],)
)

# Calcula o percentual de votos de cada candidato por bairro 
total_votos_bairro1T = df_bairros_1T.groupby('NM_BAIRRO')['QT_VOTOS'].transform('sum')
df_bairros_1T['PCT_VOTOS_BAIRRO'] = (df_bairros_1T['QT_VOTOS'] / total_votos_bairro1T) * 100.00
df_bairros_1T['PCT_VOTOS_BAIRRO'] = df_bairros_1T['PCT_VOTOS_BAIRRO'].round(2)

df_bairros_1T.head(10)

,NM_BAIRRO,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,PCT_VOTOS_BAIRRO
3,ALTO DAS MERCES,LULA,13,269,52.44
2,ALTO DAS MERCES,FLAVIO BOLSONARO,22,154,30.02
4,ALTO DAS MERCES,Nulo,96,40,7.80
0,ALTO DAS MERCES,Branco,95,17,3.31
1,ALTO DAS MERCES,ESCRITOR AUGUSTO CURY,70,12,2.34
6,ALTO DAS MERCES,RONALDO CAIADO,55,11,2.14
5,ALTO DAS MERCES,RENAN SANTOS,14,6,1.17
7,ALTO DAS MERCES,ZEMA,30,4,0.78
12,BELA VISTA,LULA,13,601,46.37
10,BELA VISTA,FLAVIO BOLSONARO,22,502,38.73


In [58]:
df_bairros_2T = (
    df_turno2.groupby(['NM_BAIRRO', 'NM_VOTAVEL', "NR_VOTAVEL"], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NM_BAIRRO', 'QT_VOTOS'], ascending=[True, False],)
)

# Calcula o percentual de votos de cada candidato por bairro 
total_votos_bairro2T = df_bairros_2T.groupby('NM_BAIRRO')['QT_VOTOS'].transform('sum')
df_bairros_2T['PCT_VOTOS_BAIRRO'] = (df_bairros_2T['QT_VOTOS'] / total_votos_bairro2T) * 100.00
df_bairros_2T['PCT_VOTOS_BAIRRO'] = df_bairros_2T['PCT_VOTOS_BAIRRO'].round(2)

df_bairros_2T.head(10)

,NM_BAIRRO,NM_VOTAVEL,NR_VOTAVEL,QT_VOTOS,PCT_VOTOS_BAIRRO
2,ALTO DAS MERCES,LULA,13,340,63.55
1,ALTO DAS MERCES,FLAVIO BOLSONARO,22,167,31.21
3,ALTO DAS MERCES,Nulo,96,23,4.30
0,ALTO DAS MERCES,Branco,95,5,0.93
6,BELA VISTA,LULA,13,754,56.69
5,BELA VISTA,FLAVIO BOLSONARO,22,530,39.85
7,BELA VISTA,Nulo,96,30,2.26
4,BELA VISTA,Branco,95,16,1.20
10,BONFIM,LULA,13,1490,53.23
9,BONFIM,FLAVIO BOLSONARO,22,1205,43.05


## Choropleth Maps do 1º Turno

**Lista os candidatos e as cores atribuidas para cada um deles**

In [66]:
opcoes_voto = [
    "Lula", "Renan Santos", "Hertz Dias", "Edmilson Costa", "Flavio Bolsonaro", "Clariana Barao", "Leonardo Avalanche", "Rui Costa Pimenta", "Zema", "Veterinário Wilson Grassi", "Ronaldo Caiado", "Escritor Augusto Cury", "Samara", "Branco", "Nulo",
]

cores_partidos = {
    13: "#ff0000", # Lula
    14: "#f0d01a", # Renan Santos
    16: "#780f07", # Hertz Dias
    21: "#fc4521", # Edmilson Costa
    22: "#0c7a0a", # Flavio Bolsonaro
    27: "#80d7ff", # Clariana Barão 
    28: "#0a1e57", # Leonardo Avalanche
    29: "#9c2308", # Rui Costa Pimenta  
    30: "#e66909", # Zema
    35: "#187031", # Veterinário Wilson Grassi
    55: "#0306a6", # Ronaldo Caiado
    70: "#077375", # Escritor Augusto Cury
    80: "#000000", # Samara Martins
    95: "#ffffff", # Branco 
    96: "#a3a3a3", # Nulo
}

**Função para atribuir paleta de cores para cada candidato**

In [67]:
def candidate_color_func(party:int):
    if party in cores_partidos.keys():
        return cores_partidos[party]
    
    # Outros? (Nao é pra ter)
    return "#ff00dd"

**Cria o mapa em SJDR**

In [ ]:
sjdr_lat_lon = [-21.1365, -44.2619]

mapa_1T_locais = folium.Map(
    location=sjdr_lat_lon,
    zoom_start=12,
    tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='© OpenStreetMap contributors',
    control_scale=True, # Mostra escala do mapa no canto inferior esquerdo
    
    # Não permite sair da região do mapa
    max_bounds=True, 
    min_lat=sjdr_lat_lon[0] + 1.0,
    max_lat=sjdr_lat_lon[0] - 1.0,
    min_lon=sjdr_lat_lon[1] + 1.0,
    max_lon=sjdr_lat_lon[1] - 1.0,
)

# Plugin de deixar o mapa em tela cheia
Fullscreen(
    position="topright",
    force_separate_button=True,
    title="Entrar no Modo Tela Cheia",
    title_cancel="Sair do Modo Tela Cheia",
).add_to(mapa_1T_locais)

**Cria uma camada para cada uma das opcoes de voto**

In [70]:
camadas_candidatos_1T = {
    opcao_voto: folium.FeatureGroup(
        name=opcao_voto,
        show=(opcao_voto == "Lula") # Deixa o Lula por padrao 
    )
    for opcao_voto in opcoes_voto
}

**Coloca os marcadores de votos em cada um dos locais de votacao, para cada uma das opcoes de voto**

In [72]:
for row in df_locais_1T.itertuples(index=False):
    lat = row.NR_LATITUDE 
    lon = row.NR_LONGITUDE 

    candidato = str(row.NM_VOTAVEL).title()
    if candidato not in camadas_candidatos_1T:
        print(f'ERRO! {candidato} não está entre os candidatos listados.')
        continue
    camada = camadas_candidatos_1T[candidato] 
    
    pct = row.PCT_VOTOS_LOCAL
    party = row.NR_VOTAVEL
    color = candidate_color_func(party)
    votos = row.QT_VOTOS
   
    nm_local = row.NM_LOCAL_VOTACAO 
    
    folium.Circle(
        location=[lat, lon],
        radius=math.sqrt(row.QT_VOTOS) * 5,
        popup=folium.Popup(
            f"<b>{candidato}</b><br>"
            f"Local: {nm_local}"
            f"Votos: {votos} ({pct}%)",
            max_width=250
        ),
        tooltip=f"{candidato}: {votos} votos ({pct}%)",
        color=color,
        fill=True,
        fill_opacity=min(0.75, 0.15 + (row.PCT_VOTOS_LOCAL / 100.0))
    ).add_to(camada)
   
# Adiciona as camadas ao mapa 
for camada in camadas_candidatos_1T.values():
    camada.add_to(mapa_1T_locais)
folium.LayerControl(collapsed=False).add_to(mapa_1T_locais)


**Exibe o mapa gerado**

In [73]:
display(mapa_1T_locais)
mapa_1T_locais.save('mapa_1t.html')

## Choropleth Maps do 2º Turno